# Pixelizator: перенос ROOT-логики в Python

Этот ноутбук повторяет основную идею `example.cpp`: анизотропная перепикселизация из гексагональной камеры в квадратную сетку.

Что делает ноутбук:
1. Читает входной txt-файл с событиями.
2. Позволяет обработать **одно событие** и построить картинку сравнения: исходная гекс-сетка vs новая квадратная сетка.
3. Считает параметры Хилласа (`length`, `width`, `theta`) для всего датасета:
   - по исходным координатам;
   - по конвертированным координатам.
4. Строит попарные гистограммы на одной канве (старые/новые координаты).
5. Включает **опциональные** режимы для сравнения подходов:
   - передача имени файла аргументом;
   - использование глобальной ориентации эллипса (из Хилласа исходного изображения);
   - динамический `R` (радиус локального окна).


In [ ]:
import argparse
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 120


In [ ]:
@dataclass
class Config:
    pixel_size: float = 1.5
    square_bin_size: float = 0.5
    x_min: float = -40.0
    x_max: float = 40.0
    y_min: float = -40.0
    y_max: float = 40.0

    neighborhood_radius: float = 7.5
    spread_factor: float = 1.5
    min_axis_size: float = 1.5
    weight_type: str = 'gauss'  # gauss | linear | quad | const

    use_global_orientation: bool = False
    use_dynamic_radius: bool = False
    dynamic_radius_base: float = 7.5
    dynamic_radius_gain: float = 0.8
    dynamic_radius_min: float = 4.0
    dynamic_radius_max: float = 14.0


def parse_args(default_file='020321.cleanout_14_7.0fix_001.txt'):
    parser = argparse.ArgumentParser(add_help=False)
    parser.add_argument('--input-file', type=str, default=default_file)
    parser.add_argument('--event-id', type=int, default=None)
    parser.add_argument('--weight-type', type=str, default='gauss')
    parser.add_argument('--use-global-orientation', action='store_true')
    parser.add_argument('--use-dynamic-radius', action='store_true')
    args, _ = parser.parse_known_args()
    return args


args = parse_args()
cfg = Config(weight_type=args.weight_type,
             use_global_orientation=args.use_global_orientation,
             use_dynamic_radius=args.use_dynamic_radius)

input_path = Path(args.input_file)
if not input_path.exists():
    input_path = Path('.') / '020321.cleanout_14_7.0fix_001.txt'

print(f'Input file: {input_path.resolve()}')
print(cfg)


In [ ]:
def read_events(filepath):
    events = []
    cur_event_id = None
    xs, ys, amps = [], [], []

    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            tokens = line.split()

            if len(tokens) == 4 and ':' in tokens[2]:
                if cur_event_id is not None:
                    events.append({'event_id': cur_event_id, 'x': np.array(xs, float), 'y': np.array(ys, float), 'amp': np.array(amps, float)})
                cur_event_id = int(tokens[0])
                xs, ys, amps = [], [], []
                continue

            if len(tokens) >= 5:
                try:
                    _, _, x, y, a = tokens[:5]
                    xs.append(float(x)); ys.append(float(y)); amps.append(float(a))
                except ValueError:
                    pass

    if cur_event_id is not None:
        events.append({'event_id': cur_event_id, 'x': np.array(xs, float), 'y': np.array(ys, float), 'amp': np.array(amps, float)})
    return events


events = read_events(input_path)
print(f'Loaded events: {len(events)}')
print('First event id:', events[0]['event_id'] if events else None)


In [ ]:
def compute_weight(weight_type, dist_aniso):
    if weight_type == 'linear':
        return 1.0 - dist_aniso if dist_aniso <= 1.0 else 0.0
    if weight_type == 'quad':
        return (1.0 - dist_aniso**2) if dist_aniso <= 1.0 else 0.0
    if weight_type == 'const':
        return 1.0 if dist_aniso <= 1.0 else 0.0
    return np.exp(-1.5 * dist_aniso**2)


def weighted_pca(x, y, w):
    wsum = np.sum(w)
    if wsum <= 0:
        return 0.0, 1.0, 1.0
    mx = np.sum(x * w) / wsum
    my = np.sum(y * w) / wsum
    dx = x - mx
    dy = y - my
    cxx = np.sum(w * dx * dx) / wsum
    cxy = np.sum(w * dx * dy) / wsum
    cyy = np.sum(w * dy * dy) / wsum
    trace = cxx + cyy
    det_term = np.sqrt(max((cxx - cyy)**2 + 4 * cxy**2, 0.0))
    l1 = max(0.5 * (trace + det_term), 0.0)
    l2 = max(0.5 * (trace - det_term), 0.0)
    theta = 0.5 * np.arctan2(2 * cxy, cxx - cyy)
    major = np.sqrt(l1)
    minor = np.sqrt(l2)
    if major < minor:
        major, minor = minor, major
        theta += np.pi / 2
    return theta, major, minor


def calculate_local_pca(x_all, y_all, amp_all, x0, y0, radius, min_axis_size=1.5):
    dist = np.sqrt((x_all - x0)**2 + (y_all - y0)**2)
    m = (dist <= radius) & (amp_all > 0)
    if not np.any(m):
        return 0.0, min_axis_size, min_axis_size
    theta, major, minor = weighted_pca(x_all[m], y_all[m], amp_all[m])
    major = max(major, min_axis_size)
    minor = max(minor, min_axis_size)
    return theta, major, minor


def dynamic_radius(local_amp, cfg):
    r = cfg.dynamic_radius_base + cfg.dynamic_radius_gain * np.log1p(max(local_amp, 0.0))
    return float(np.clip(r, cfg.dynamic_radius_min, cfg.dynamic_radius_max))


def event_to_square_grid(x, y, amp, cfg):
    xbins = int((cfg.x_max - cfg.x_min) / cfg.square_bin_size)
    ybins = int((cfg.y_max - cfg.y_min) / cfg.square_bin_size)
    gx = np.linspace(cfg.x_min, cfg.x_max, xbins, endpoint=False) + cfg.square_bin_size / 2
    gy = np.linspace(cfg.y_min, cfg.y_max, ybins, endpoint=False) + cfg.square_bin_size / 2
    xx, yy = np.meshgrid(gx, gy, indexing='ij')
    grid = np.zeros((xbins, ybins), dtype=float)

    global_theta, _, _ = weighted_pca(x, y, amp)

    for i in range(len(x)):
        if not (cfg.x_min <= x[i] <= cfg.x_max and cfg.y_min <= y[i] <= cfg.y_max):
            continue

        radius = cfg.neighborhood_radius
        if cfg.use_dynamic_radius:
            radius = dynamic_radius(amp[i], cfg)

        theta, major, minor = calculate_local_pca(x, y, amp, x[i], y[i], radius, cfg.min_axis_size)
        if cfg.use_global_orientation:
            theta = global_theta

        major *= cfg.spread_factor
        minor *= cfg.spread_factor

        dx = xx - x[i]
        dy = yy - y[i]
        c, s = np.cos(-theta), np.sin(-theta)
        dxr = dx * c - dy * s
        dyr = dx * s + dy * c
        dist = np.sqrt((dxr / major)**2 + (dyr / minor)**2)
        mask = dist <= 1.0

        if not np.any(mask):
            ix = int(np.clip(np.floor((x[i] - cfg.x_min) / cfg.square_bin_size), 0, xbins - 1))
            iy = int(np.clip(np.floor((y[i] - cfg.y_min) / cfg.square_bin_size), 0, ybins - 1))
            grid[ix, iy] += amp[i]
            continue

        w = np.zeros_like(dist)
        w[mask] = np.vectorize(lambda d: compute_weight(cfg.weight_type, d))(dist[mask])
        sw = w.sum()
        if sw > 0:
            grid += amp[i] * (w / sw)

    return gx, gy, grid


def hillas_parameters(x, y, amp):
    m = amp > 0
    if np.count_nonzero(m) < 2:
        return np.nan, np.nan, np.nan, np.nan
    theta, major, minor = weighted_pca(x[m], y[m], amp[m])
    size = float(np.sum(amp[m]))
    return major, minor, np.degrees(theta), size


In [ ]:
if not events:
    raise RuntimeError('No events were loaded.')

if args.event_id is None:
    event = events[0]
else:
    matched = [e for e in events if e['event_id'] == args.event_id]
    if not matched:
        raise ValueError(f'Event id {args.event_id} not found')
    event = matched[0]

print('Selected event id:', event['event_id'], 'hits:', len(event['x']))

gx, gy, grid = event_to_square_grid(event['x'], event['y'], event['amp'], cfg)
orig_int = float(np.sum(event['amp']))
new_int = float(np.sum(grid))
print(f'Integral original={orig_int:.6f}, converted={new_int:.6f}, diff={abs(orig_int-new_int):.6e}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
sc = axes[0].scatter(event['x'], event['y'], c=event['amp'], cmap='inferno', s=120, marker='h', edgecolor='k', linewidth=0.2)
axes[0].set_title(f"Event {event['event_id']} - original (hex centers)")
axes[0].set_xlabel('X [cm]'); axes[0].set_ylabel('Y [cm]')
axes[0].set_xlim(cfg.x_min, cfg.x_max); axes[0].set_ylim(cfg.y_min, cfg.y_max)
axes[0].set_aspect('equal')
fig.colorbar(sc, ax=axes[0], fraction=0.046, pad=0.04, label='Amplitude')

im = axes[1].imshow(grid.T, origin='lower', extent=[cfg.x_min, cfg.x_max, cfg.y_min, cfg.y_max], cmap='inferno', aspect='equal')
axes[1].set_title('Converted square grid')
axes[1].set_xlabel('X [cm]'); axes[1].set_ylabel('Y [cm]')
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04, label='Amplitude')
plt.show()


In [ ]:
length_old, width_old, theta_old, size_old = [], [], [], []
length_new, width_new, theta_new, size_new = [], [], [], []

for ev in events:
    lo, wo, to, so = hillas_parameters(ev['x'], ev['y'], ev['amp'])
    gx, gy, g = event_to_square_grid(ev['x'], ev['y'], ev['amp'], cfg)
    ix, iy = np.where(g > 0)
    if len(ix) > 1:
        x_new = gx[ix]; y_new = gy[iy]; a_new = g[ix, iy]
        ln, wn, tn, sn = hillas_parameters(x_new, y_new, a_new)
    else:
        ln, wn, tn, sn = np.nan, np.nan, np.nan, np.nan

    length_old.append(lo); width_old.append(wo); theta_old.append(to); size_old.append(so)
    length_new.append(ln); width_new.append(wn); theta_new.append(tn); size_new.append(sn)

length_old = np.array(length_old); width_old = np.array(width_old); theta_old = np.array(theta_old); size_old = np.array(size_old)
length_new = np.array(length_new); width_new = np.array(width_new); theta_new = np.array(theta_new); size_new = np.array(size_new)

fig, axes = plt.subplots(1, 4, figsize=(20, 4), constrained_layout=True)
for ax, (name, arr_old, arr_new) in zip(axes, [
    ('length', length_old, length_new),
    ('width', width_old, width_new),
    ('theta [deg]', theta_old, theta_new),
    ('size', size_old, size_new),
]):
    m1 = np.isfinite(arr_old); m2 = np.isfinite(arr_new)
    vals = np.concatenate([arr_old[m1], arr_new[m2]]) if (np.any(m1) and np.any(m2)) else np.array([0, 1])
    bins = np.linspace(vals.min(), vals.max(), 35)
    ax.hist(arr_old[m1], bins=bins, alpha=0.55, label='original coords')
    ax.hist(arr_new[m2], bins=bins, alpha=0.55, label='converted coords')
    ax.set_title(f'Hillas {name}')
    ax.set_xlabel(name); ax.set_ylabel('N events'); ax.legend()
plt.show()


In [ ]:
# Опциональное сравнение подходов: базовый / global orientation / dynamic R

def collect_hillas(events, cfg):
    out = {'length': [], 'width': [], 'theta': [], 'size': []}
    for ev in events:
        gx, gy, g = event_to_square_grid(ev['x'], ev['y'], ev['amp'], cfg)
        ix, iy = np.where(g > 0)
        if len(ix) > 1:
            x_new = gx[ix]; y_new = gy[iy]; a_new = g[ix, iy]
            l, w, t, s = hillas_parameters(x_new, y_new, a_new)
        else:
            l, w, t, s = np.nan, np.nan, np.nan, np.nan
        out['length'].append(l); out['width'].append(w); out['theta'].append(t); out['size'].append(s)
    for k in out:
        out[k] = np.array(out[k], dtype=float)
    return out

cfg_base = Config()
cfg_global = Config(use_global_orientation=True)
cfg_dyn = Config(use_dynamic_radius=True)

h_base = collect_hillas(events, cfg_base)
h_global = collect_hillas(events, cfg_global)
h_dyn = collect_hillas(events, cfg_dyn)

fig, axes = plt.subplots(1, 4, figsize=(20, 4), constrained_layout=True)
for ax, key in zip(axes, ['length', 'width', 'theta', 'size']):
    for label, arr in [('base', h_base[key]), ('global-orient', h_global[key]), ('dynamic-R', h_dyn[key])]:
        m = np.isfinite(arr)
        ax.hist(arr[m], bins=30, alpha=0.35, label=label)
    ax.set_title(f'Converted: {key}')
    ax.legend()
plt.show()
